# Rebel Foods – Cloud Kitchen P&L Analysis
**Python:** 3.10+  
**Packages:** pandas==2.2, plotly==5.22, openpyxl==3.1, numpy==1.26

---

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')


print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
import plotly; print(f'plotly  : {plotly.__version__}')
import openpyxl; print(f'openpyxl: {openpyxl.__version__}')

## 1. Load & Clean Data

In [ ]:
df = pd.read_excel('Kittchen_PNL_Data.xlsx', sheet_name='Sheet 1 - stores', header=1)


df['GM%']       = (df['GROSS MARGIN']   / df['NET REVENUE'] * 100).round(2)
df['CM']        = df['GROSS MARGIN']
df['CM%']       = df['GM%']
df['EBITDA']    = df['KITCHEN EBITDA']
df['EBITDA%']   = (df['EBITDA']         / df['NET REVENUE'] * 100).round(2)
df['VARIANCE%'] = (df['VARIANCE']       / df['NET REVENUE'] * 100).round(4)


bins   = [0, 1_500_000, 2_500_000, 3_500_000, 4_500_000, float('inf')]
labels = ['(a) Below INR 15 lacs','(b) INR 15 to 25 lacs',
          '(c) INR 25 to 35 lacs','(d) INR 35 to 45 lacs','(e) Above INR 45 lacs']
df['REV_BUCKET'] = pd.cut(df['NET REVENUE'], bins=bins, labels=labels)


vbins   = [0, 2, 3, 5, float('inf')]
vlabels = ['(a) Var < 2%','(b) Var 2% to 3%','(c) Var 3% to 5%','(d) Var > 5%']
df['VAR_BUCKET'] = pd.cut(df['VARIANCE%'], bins=vbins, labels=vlabels)

month_order = ['Oct-2023','Nov-2023','Dec-2023','Jan-2024','Feb-2024','Mar-2024']
df['MONTH'] = pd.Categorical(df['MONTH'], categories=month_order, ordered=True)

print('Shape:', df.shape)
df.head(3)

## 2. Data Overview

In [ ]:
print('Unique Stores :', df['STORE'].nunique())
print('Unique Cities :', df['CITY'].unique())
print('Months        :', month_order)
print('Revenue Cohorts:', df['REVENUE COHORT'].unique())
print('EBITDA Category:', df['EBITDA CATEGORY'].unique())
print()
df[['NET REVENUE','GM%','EBITDA','EBITDA%','VARIANCE%']].describe().round(2)

## 3. Dashboard 1 – Kitchen Level PNL

In [ ]:

trend = df.groupby('MONTH', observed=True)['NET REVENUE'].sum().reset_index()
fig = px.bar(trend, x='MONTH', y='NET REVENUE',
             title='Net Revenue by Month',
             color_discrete_sequence=['#F5C518'])
fig.show()

In [ ]:

city = df.groupby('CITY')['NET REVENUE'].sum().reset_index().sort_values('NET REVENUE', ascending=False)
fig2 = px.pie(city, values='NET REVENUE', names='CITY',
              title='Revenue Share by City', hole=0.4)
fig2.show()

In [ ]:

ebitda_trend = df.groupby('MONTH', observed=True)['EBITDA%'].mean().reset_index()
fig3 = px.line(ebitda_trend, x='MONTH', y='EBITDA%',
               title='Avg EBITDA% Trend', markers=True,
               color_discrete_sequence=['#E94560'])
fig3.add_hline(y=0, line_dash='dash', line_color='red', annotation_text='Break-even')
fig3.show()

In [ ]:

cat_cnt = df['EBITDA CATEGORY'].value_counts().reset_index()
cat_cnt.columns = ['Category', 'Count']
fig4 = px.bar(cat_cnt, x='Category', y='Count', color='Category',
              title='EBITDA +ve vs -ve Store-Months',
              color_discrete_map={'EBITDA +ve': '#00B074', 'EBITDA -ve': '#E94560'})
fig4.show()

In [ ]:

snapshot = df[['STORE','MONTH','NET REVENUE','GM%','CM%','EBITDA','EBITDA%','REVENUE COHORT','EBITDA CATEGORY']]
snapshot = snapshot.sort_values(['STORE','MONTH'])
print(snapshot.head(12).to_string(index=False))

## 4. Dashboard 2 – Variance Level PNL

In [ ]:

pivot_a = (
    df.groupby(['REV_BUCKET','MONTH'], observed=True)['VARIANCE%']
    .mean()
    .round(2)
    .reset_index()
    .pivot(index='REV_BUCKET', columns='MONTH', values='VARIANCE%')
)
pivot_a.loc['Grand total'] = pivot_a.mean()
print('Sub-dashboard A — Avg Variance% by Revenue Category:')
print(pivot_a.applymap(lambda x: f'{x:.2f}%' if pd.notna(x) else '—'))

In [ ]:

pivot_b = (
    df.groupby(['REV_BUCKET','MONTH'], observed=True)['STORE']
    .nunique()
    .reset_index()
    .pivot(index='REV_BUCKET', columns='MONTH', values='STORE')
)
pivot_b.loc['Grand total'] = pivot_b.sum()
print('Sub-dashboard B — Store Count by Revenue Bucket:')
print(pivot_b.fillna(0).astype(int))

In [ ]:

pivot_num = (
    df.groupby(['REV_BUCKET','MONTH'], observed=True)['VARIANCE%']
    .mean()
    .round(2)
    .reset_index()
    .pivot(index='REV_BUCKET', columns='MONTH', values='VARIANCE%')
)
fig_h = go.Figure(go.Heatmap(
    z=pivot_num.values,
    x=list(pivot_num.columns),
    y=list(pivot_num.index),
    colorscale='YlOrRd',
    text=[[f'{v:.2f}%' for v in row] for row in pivot_num.values],
    texttemplate='%{text}'
))
fig_h.update_layout(title='Avg Variance% — Revenue Category × Month')
fig_h.show()

## 5. Additional Insights

In [ ]:

top10 = df.groupby('STORE')['NET REVENUE'].sum().nlargest(10).reset_index()
fig_t = px.bar(top10, x='STORE', y='NET REVENUE', title='Top 10 Stores by Revenue',
               color_discrete_sequence=['#F5C518'])
fig_t.update_xaxes(tickangle=30)
fig_t.show()

In [ ]:

fig_box = px.box(df, x='ZONE MAPPING', y='VARIANCE%', color='ZONE MAPPING',
                 title='Variance% Distribution by Zone', points='outliers')
fig_box.show()

In [ ]:

flip = df.groupby('STORE')['EBITDA CATEGORY'].nunique()
print('Stores with inconsistent EBITDA category across months:', (flip > 1).sum())